In [ ]:
#----------------------------------------------------------------------------------
# 1. prepare:import, define model, basic configuration
import torch
from torch import nn
from torch.nn import Module
from torchvision.datasets import mnist
from torch.nn import CrossEntropyLoss
from torch.optim import SGD
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
import numpy as np
import os
import copy
import math

class Model(Module):
    def __init__(self):
        super(Model,self).__init__()
        self.conv1=nn.Conv2d(1,6,5)
        self.relu1=nn.ReLU()
        self.maxpool1=nn.MaxPool2d(2)
        self.conv2=nn.Conv2d(6,16,5)
        self.relu2=nn.ReLU()
        self.maxpool2=nn.MaxPool2d(2)
        self.fc1=nn.Linear(256,120)
        self.relu3=nn.ReLU()
        self.fc2=nn.Linear(120,84)
        self.relu4=nn.ReLU()
        self.fc3=nn.Linear(84,10)

    def forward(self, x):
        y = self.conv1(x)
        y = self.relu1(y)
        y = self.maxpool1(y)
        y = self.conv2(y)
        y = self.relu2(y)
        y = self.maxpool2(y)
        y = y.view(y.shape[0], -1)
        y = self.fc1(y)
        y = self.relu3(y)
        y = self.fc2(y)
        y = self.relu4(y)
        y = self.fc3(y)
        return y

device='cpu'
batch_size=256
train_dataset=mnist.MNIST(root='./train', train=True, transform=ToTensor(), download=True)
test_dataset=mnist.MNIST(root='./test', train=False, transform=ToTensor(), download=True)
train_loader=DataLoader(train_dataset, batch_size=batch_size)
test_loader=DataLoader(test_dataset, batch_size=batch_size)
model=Model().to(device)
sgd=SGD(model.parameters(), lr=1e-1) #gradient descent
loss_fn=CrossEntropyLoss() #loss function
all_epoch=10

#-----------------------------------------------------------------------------------
#2. measure accuacry function and TRAIN to get a model
def measure_acc(model):
  all_correct_number=0
  all_sample_number=0
  model.eval()
  for idx,(test_x, test_label) in enumerate (test_loader):
    test_x = test_x.to(device)
    test_label=test_label.to(device)
    predict_y=model(test_x.float()).detach()
    predict_y=torch.argmax(predict_y, dim=-1)
    current_correct_number=predict_y==test_label
    all_correct_number += current_correct_number.sum().item()
    all_sample_number += test_x.shape[0]
  acc= all_correct_number / all_sample_number
  return(acc)

for current_epoch in range(all_epoch):
  model.train()
  for idx, (train_x,train_label) in enumerate (train_loader):
    train_x=train_x.to(device)
    train_label=train_label.to(device)
    sgd.zero_grad()               #clear gradients
    predict_y=model(train_x.float())       #predict result
    loss=loss_fn(predict_y,train_label.long())  #mark/compute loss
    loss.backward()               #get gradients
    sgd.step()                 #write weights

#--------------------------------------------------------------------------------------
#3.calibration：run 10 batches data, and record each layer max value, in total 5 number
@torch.no_grad()
def calibrate(model, loader, n_batches=10):
    model.eval()
    maxes = {"in": 0.0, "c1": 0.0, "c2": 0.0, "f1": 0.0, "f2": 0.0}

    for i, (x, _) in enumerate(loader):
        maxes["in"] = max(maxes["in"], x.max().item())        # record max input

        y = model.relu1(model.conv1(x))
        maxes["c1"] = max(maxes["c1"], y.max().item())        # record max value after conv1 + ReLU
        y = model.maxpool1(y)

        y = model.relu2(model.conv2(y))
        maxes["c2"] = max(maxes["c2"], y.max().item())        # record max value after conv2 + ReLU
        y = model.maxpool2(y)

        y = y.view(y.shape[0], -1)

        y = model.relu3(model.fc1(y))
        maxes["f1"] = max(maxes["f1"], y.max().item())        # record max value after fc1 + ReLU

        y = model.relu4(model.fc2(y))
        maxes["f2"] = max(maxes["f2"], y.max().item())        # record max value after fc2 + ReLU

        if i + 1 >= n_batches:
            break

    return maxes
#-----------------------------------------------------------------------------------
#4. calculate scale（or called exponent) and shift（which used to requantization)
def pow2_exp(max_val, qmax): # calculate exponent function, scale * qmax >= max_val（n < log2(qmax/max))
    return math.floor(math.log2(qmax / max_val))


def get_exponents(model, maxes, bits):  #calculate activation and weight exponent
    act_qmax = 2 ** bits - 1          # INT8 -> 255, activation have no negative
    w_qmax   = 2 ** (bits - 1) - 1    # INT8 -> 127, weight have negative

    # activation exponent(from calibration)
    act_exp = {}
    for name, v in maxes.items():
      act_exp[name] = pow2_exp(v, act_qmax)      # act_exp should like {'in': 7, 'c1': 5, 'c2': 4, 'f1': 3, 'f2': 3}

    # weight exponent(direct from weight)
    w_exp = {}
    for name in ["conv1", "conv2", "fc1", "fc2", "fc3"]:
        w = getattr(model, name).weight
        w_exp[name] = pow2_exp(w.abs().max().item(), w_qmax) # w_exp should like {'conv1': 7, 'conv2': 8, 'fc1': 8, 'fc2': 8, 'fc3': 8}

    return act_exp, w_exp

def get_shifts(act_exp, w_exp): #calculate shift
    layers = [
        # (layer name, input exponent name, output exponent name)
        ("conv1", "in", "c1"),
        ("conv2", "c1", "c2"),
        ("fc1",   "c2", "f1"),
        ("fc2",   "f1", "f2"),
    ]

    shifts = {}
    for name, in_name, out_name in layers:
        a = act_exp[in_name]     # activation(input) exponent
        b = w_exp[name]          # weight exponent
        c = act_exp[out_name]    # output(next layer input) exponent
        shifts[name] = a + b - c
        print(f"{name}: a={a}, b={b}, c={c}  ->  shift={shifts[name]}")

    return shifts

bits = 4  #IMPORTANT CHANGE HERE !!!
maxes = calibrate(model, train_loader)
act_exp, w_exp = get_exponents(model, maxes, bits=bits)
print("activation exponent:", act_exp)
print("weight exponent:", w_exp)
shifts = get_shifts(act_exp, w_exp)

#------------------------------------------------------------------------------------
#5. integer model and inference
class QModel(Model): # Quantised model: same structure as Model, forward adds quantisation steps ==========
    def __init__(self, bits):
        super().__init__()
        self.qmax = 2 ** bits - 1               # INT8 -> 255

    def requant(self, y, shift):
        return torch.clamp(torch.floor(y / 2 ** shift), 0, self.qmax)

    def forward(self, x):

        x = torch.clamp(torch.round(x.double() * 2 ** self.in_exp), 0, self.qmax) # Input quantisation: multiply by 2^a, round, clamp to [0, 255]

        y = self.maxpool1(self.requant(self.relu1(self.conv1(x)), self.conv1.shift))
        y = self.maxpool2(self.requant(self.relu2(self.conv2(y)), self.conv2.shift))
        y = y.view(y.shape[0], -1)
        y = self.requant(self.relu3(self.fc1(y)), self.fc1.shift)
        y = self.requant(self.relu4(self.fc2(y)), self.fc2.shift)
        return self.fc3(y)                       # Last layer: no ReLU, no requantisation


qm = QModel(bits).to(device)   # Create the model called 'qm'
qm.load_state_dict(model.state_dict())          # Copy the floating-point weights first
qm.double()                                     # Use float64 so integer arithmetic stays exact
qm.in_exp = act_exp["in"]                       # Exponent used for input quantisation

with torch.no_grad():
    for name, in_name in [("conv1", "in"), ("conv2", "c1"), ("fc1", "c2"), ("fc2", "f1"), ("fc3", "f2")]:
        layer = getattr(qm, name)
        a = act_exp[in_name]                     # Exponent of the layer input
        b = w_exp[name]                          # Exponent of the weights
        shift = shifts.get(name)                 # fc3 has no shift, so this is None

        w_int = torch.clamp(torch.round(layer.weight * 2 ** b), -2 ** (bits - 1), 2 ** (bits - 1) - 1)
        b_int = torch.round(layer.bias * 2 ** (a + b))
        if shift is not None:
            b_int = b_int + 2 ** (shift - 1)     # Fold the rounding offset (half of 2^shift) into the bias

        layer.weight.copy_(w_int)                # Write back the integer weights
        layer.bias.copy_(b_int)                  # Write back the integer bias
        layer.shift = shift                      # Store the shift on the layer as well


# ========== Measure accuracy with the existing measure_acc ==========
print("FP32 accuracy:", measure_acc(model))
print("INT accuracy:", measure_acc(qm))
print(qm.conv1.weight)                    # Quick check: should be all integers
print(qm.conv1.bias)
print(qm.conv1.shift)

print('finish training')

activation exponent: {'in': 3, 'c1': 0, 'c2': -1, 'f1': 0, 'f2': 0}
weight exponent: {'conv1': 2, 'conv2': 3, 'fc1': 5, 'fc2': 4, 'fc3': 3}
conv1: a=3, b=2, c=0  ->  shift=5
conv2: a=0, b=3, c=-1  ->  shift=4
fc1: a=-1, b=5, c=0  ->  shift=4
fc2: a=0, b=4, c=0  ->  shift=4
FP32 accuracy: 0.9826
INT accuracy: 0.974
Parameter containing:
tensor([[[[ 2.,  2.,  1., -1., -2.],
          [ 3.,  4.,  2.,  1., -0.],
          [ 3.,  4.,  4.,  1.,  0.],
          [ 0.,  1.,  2.,  2.,  1.],
          [-0., -0.,  1.,  0., -0.]]],


        [[[-1., -0., -1., -0.,  1.],
          [-0., -1., -0.,  1.,  1.],
          [-1., -1., -1.,  0.,  2.],
          [-1., -1.,  1.,  1.,  2.],
          [-1., -1., -0.,  2.,  1.]]],


        [[[ 0.,  1., -0., -1.,  1.],
          [-1., -1., -0.,  1.,  1.],
          [-0., -1.,  1.,  2.,  2.],
          [-0., -0.,  2.,  2.,  2.],
          [ 1.,  0.,  1.,  2.,  1.]]],


        [[[ 0., -0., -0.,  1., -0.],
          [ 1., -0.,  0.,  1., -0.],
          [-1.,  0., 